# 01 · Define & Explore — multi-state theory + the two-state hello-world

**Standard slot:** *define & explore.* **For Project 22 this means:** understand what a
conformational switch *is* (ONE sequence compatible with TWO backbones, toggled by a trigger), pin
down the metrics, define your two states + trigger, and run the **mock hello-world** end-to-end:
one shared sequence → predict **both** states → compute the energy gap (D0).

Run `00_setup.ipynb` first in this session. The mock backend runs anywhere (no GPU); switch to the
real RFdiffusion/MPNN/AF2 backends on Colab/A100.

## What a conformational switch is (and why it's hard)

A switch is **one amino-acid sequence that is compatible with two distinct backbones** — state A and
state B — and interconverts between them when a **trigger** fires (pH / ligand / light / temperature).
The Baker lab's **LOCKR** showed de novo switches are possible, but they are rare:

- a sequence that fits state A well usually fits state B **badly**, and
- our structure predictors (AF2) typically return a **single** dominant state, so even *confirming*
  the switch in silico is genuinely hard.

So the whole project is a search for the unusual sequence that satisfies **both** states, plus an
honest account of how often *anything* does.

## The metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| per-state scRMSD | Å | designed-vs-predicted Cα-RMSD for ONE state (< 2 Å = foldable to that shape) | the protein toggles / state is populated |
| per-state pLDDT | 0–100 | local confidence of that state's refold | thermostability / ΔG / "this state exists" |
| state energy gap | relative units (**NOT kcal/mol**) | how far apart the two states sit (proxy from per-state fit) | a real ΔΔG |
| per-state MPNN score | — | multi-state MPNN's fit to each backbone (lower = better) | binding/function |

A switch must satisfy **per-state scRMSD < 2 Å for A AND B** *and* land **inside** the energy-gap
band `[SWITCH_GAP_MIN, SWITCH_GAP_MAX]`: too large a gap ⇒ the high-energy state is never populated
(no switch); too small ⇒ the states are indistinct (no defined OFF/ON). Write your own one-paragraph
definitions in `D0`, including the "does not mean" column — that is where the published mistakes live.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Define your two states + trigger

Edit `data/inputs/two_state_def.txt` (the Phase-0 template) with your **two states + trigger +
measurable success criteria + controls**. The notebook reads the **topology** and **trigger** from
there; the two backbones themselves are *generated* in notebook 02. Below we just transcribe the
two knobs the campaign needs (keep them in sync with the file).

In [ ]:
import multistate_tools as ms

# Transcribe these from data/inputs/two_state_def.txt (your D0 problem statement).
TOPOLOGY = "hinge"     # one of ms.TOPOLOGIES
TRIGGER  = "pH"        # one of ms.TRIGGERS
LENGTH   = 100         # same length in both states (tied multi-state design)

assert TOPOLOGY in ms.TOPOLOGIES, ms.TOPOLOGIES
assert TRIGGER in ms.TRIGGERS, ms.TRIGGERS
print("topologies:", ms.TOPOLOGIES)
print("triggers   :", ms.TRIGGERS)
print(f"this design: {TRIGGER}-triggered toggle between two {TOPOLOGY} states, L={LENGTH}")
print("per-state bars: scRMSD <", ms.SELF_CONSISTENT_SCRMSD, "A, pLDDT >=", ms.SWITCH_PLDDT)
print("switchable gap band:", (ms.SWITCH_GAP_MIN, ms.SWITCH_GAP_MAX), "(relative units, NOT kcal/mol)")

## Mock hello-world: one sequence → two states → energy gap

Using the **deterministic mock backend** (synthetic, labelled `EXAMPLE_DATA` — **never** present as
real designs), we exercise the entire plumbing: generate the two state backbones, design ONE shared
sequence with multi-state MPNN, predict **both** states from that one sequence, and compute the
energy gap. On Colab/A100 you flip `tool="mock"` → the real backends.

In [ ]:
# 1) the two state backbones (mock: placeholders; on Colab -> RFdiffusion x2)
tsd = ms.generate_two_states(topology=TOPOLOGY, trigger=TRIGGER, tool="mock", length=LENGTH, seed=0)
print("state A:", tsd.state_a.state, tsd.state_a.topology, "L=", tsd.state_a.length, "| synthetic=", tsd.state_a.synthetic)
print("state B:", tsd.state_b.state, tsd.state_b.topology, "L=", tsd.state_b.length, "| synthetic=", tsd.state_b.synthetic)
print("trigger:", tsd.trigger, "-", tsd.trigger_detail)

# 2) ONE shared sequence compatible with BOTH backbones (mock multi-state MPNN)
seq = ms.multistate_mpnn(tsd.state_a, tsd.state_b, n=1, tool="mock", seed=0)[0]
print("\nshared sequence:", seq.design_id)
print("  per-state MPNN score  A:", seq.mpnn_score_a, " B:", seq.mpnn_score_b, "(lower = better fit)")

# 3) predict BOTH states from that ONE sequence
pa = ms.af2_predict_state(seq.sequence, tsd.state_a, tool="mock")
pb = ms.af2_predict_state(seq.sequence, tsd.state_b, tool="mock")
print("\nstate A  scRMSD:", pa.scrmsd_to_state, "A  pLDDT:", pa.plddt)
print("state B  scRMSD:", pb.scrmsd_to_state, "A  pLDDT:", pb.plddt)

# 4) the energy gap between the two predicted states
eg = ms.energy_gap(pa, pb)
print("\nenergy gap:", eg.gap, "| favored:", eg.favored_state, "| switchable:", eg.switchable)
print("note:", eg.note)
print("\n[mock = SYNTHETIC EXAMPLE_DATA — not a real switch]")

### Read the hello-world honestly

Notice (in the synthetic numbers) the pattern the real problem has: state A — the "designed-for"
state — tends to fit better than state B, and the gap is small and noisy. That is the multi-state
difficulty in miniature. A *real* candidate must pass **both** per-state bars **and** sit inside the
switchable band — and you must still worry that AF2 only ever modeled one of the states.

## Visualize a state (py3Dmol)
Once you have real predicted PDBs (notebook 04), eyeball state A and state B side by side.

In [ ]:
import py3Dmol

def show_pdb(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=500, height=400)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after real predictions):
# show_pdb("results/two_state/state_A_pred.pdb")
# show_pdb("results/two_state/state_B_pred.pdb")
print("show_pdb(pdb_path) ready — compare state A vs state B once you have real models.")

## D0 checklist
- [ ] One-paragraph definition of **per-state scRMSD** and the **energy gap** — *with* what each does **not** mean (gap ≠ ΔΔG; low scRMSD ≠ the switch toggles).
- [ ] `data/inputs/two_state_def.txt` filled in: two states + trigger + **measurable** success criteria + the three controls.
- [ ] Reproduced hello-world: one shared sequence → two per-state predictions → energy gap (screenshot/printout).
- [ ] Every accession in `data/README.md` verified on RCSB.
- [ ] `LOG.md` entry: date, backend (mock here), seed.

**Next:** `02_generate.ipynb` — generate the two backbones + multi-state MPNN shared-sequence pool.